In [29]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2, MobileNetV3Large, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import mixed_precision

import numpy as np
import os
import gc
import time
import shutil

import pandas as pd
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

MODELS = {
    "MobileNetV3Large": MobileNetV3Large
}


TensorFlow version: 2.10.1
GPU disponible: []


In [30]:
import tensorflow as tf
import numpy as np
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

IMG_SIZE = 224
TEST_DIR = "dataset-entrenamiento/test"

# ── Cargar dataset de test ────────────────────────────────
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    shuffle=False,
    label_mode='categorical'
)
class_names = test_ds.class_names
print(f"Clases: {class_names}")

X_test = np.concatenate([imgs.numpy() for imgs, _ in test_ds], axis=0)
y_true = np.concatenate([
    np.argmax(lbls.numpy(), axis=1) for _, lbls in test_ds
], axis=0)
print(f"Test set: {X_test.shape} | {len(y_true)} imágenes")

# ── Cargar modelo Keras en vez de TFLite ──────────────────
model = tf.keras.models.load_model("MobileNetV3Large.keras")

# ── Inferencia ────────────────────────────────────────────
predictions = model.predict(X_test, batch_size=32, verbose=1)
y_pred = np.argmax(predictions, axis=1)

# ── Métricas ──────────────────────────────────────────────
print(f"\nAccuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1 Score  : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")

Found 21 files belonging to 2 classes.
Clases: ['español', 'maya']
Test set: (21, 224, 224, 3) | 21 imágenes
1/1 [==============================] - 7s 7s/step

Accuracy  : 0.0000
Precision : 0.0000
Recall    : 0.0000
F1 Score  : 0.0000


## 2. Configuración GPU

In [31]:
# Growth dinámico de VRAM — evita que TF tome toda la memoria
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU configurada con memory growth: {gpus}")
    except RuntimeError as e:
        print(e)

# Mixed Precision — reduce VRAM y acelera en RTX
mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision: mixed_float16 activado")

Mixed precision: mixed_float16 activado


## 3. Configuración general

In [32]:
IMG_SIZE     = 224
EPOCHS       = 20
NUM_CLASSES  = 2       # ← cambiar según cantidad de idiomas

TRAIN_DIR    = "dataset-entrenamiento/train"
VAL_DIR      = "dataset-entrenamiento/val"
TEST_DIR     = "dataset-entrenamiento/test"

# Directorio donde se guardarán los espectrogramas aumentados
#AUG_DIR      = "train/augmented"
AUG_DIR = "dataset-entrenamiento/train"

# Batch size por modelo (ajustado a VRAM)
BATCH_SIZES = {
    "MobileNetV2":    64,
    "MobileNetV3Large": 32,
    "EfficientNetB0": 16
}
AUTOTUNE   = tf.data.AUTOTUNE

class_names = sorted([
    d for d in os.listdir(AUG_DIR)
    if os.path.isdir(os.path.join(AUG_DIR, d))
])
NUM_CLASSES = len(class_names)   # sobreescribe el valor manual
print(f"Clases detectadas: {class_names}")
print(f"NUM_CLASSES      : {NUM_CLASSES}")

# Definir manualmente si ya tienes el augmented generado

# Obtener class_names del directorio directamente
class_names = sorted(os.listdir(AUG_DIR))
print(f"Clases detectadas: {class_names}")
print(os.path.exists(AUG_DIR)) 

def predict_tflite(interpreter, dataset):
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    all_predictions = []
    
    for images, _ in dataset:
        for i in range(images.shape[0]):
            img = np.expand_dims(images[i], axis=0).astype(np.float32)
            interpreter.set_tensor(input_details[0]['index'], img)
            interpreter.invoke()
            output_data = interpreter.get_tensor(output_details[0]['index'])
            all_predictions.append(output_data[0])
            
    return np.array(all_predictions)
    

Clases detectadas: ['español', 'maya']
NUM_CLASSES      : 2
Clases detectadas: ['español', 'maya']
True


## 10. Evaluación detallada con reporte Excel

Carga todos los modelos `.keras` guardados y genera:
- Accuracy, Precision, Recall, F1 Score
- Tiempo de inferencia
- Matriz de confusión por modelo
- Reporte de clasificación por modelo
- Archivo `model_comparison.xlsx`

In [34]:
#MODELS_TFLITE = {name: f"{name}_fp16.tflite" for name in MODELS.keys()}
MODELS_TFLITE = {name: f"{name}_normal.tflite" for name in MODELS.keys()}

mis_clases = ['español', 'maya']

base_test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    class_names=mis_clases,
    batch_size=32,
    shuffle=False,
    label_mode='categorical'
)

eval_class_names = base_test_ds.class_names
print(f"Clases de evaluación: {eval_class_names}")

test_ds_eval = base_test_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# Cargar todo el test set en RAM una sola vez (inferencia más rápida)
X_test = np.concatenate([images.numpy() for images, _ in test_ds_eval], axis=0)
y_true = np.concatenate([
    np.argmax(labels.numpy(), axis=1) for _, labels in test_ds_eval
], axis=0)
print(f"Test set cargado: {X_test.shape} | Etiquetas: {y_true.shape}")

eval_results = []
excel_writer = pd.ExcelWriter("model_comparison.xlsx", engine="openpyxl")

for model_name, model_path in MODELS_TFLITE.items():

    if not os.path.exists(model_path):
        print(f"  ⚠️  {model_path} no encontrado, saltando...")
        continue

    print(f"\n{'='*55}")
    print(f"  Evaluando TFLite: {model_name}")
    print(f"{'='*55}")

    # Cargar interpreter
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_dtype    = input_details[0]['dtype']

    # Inferencia imagen por imagen
    start = time.time()
    predictions_list = []

    for i in range(len(X_test)):
        img = np.expand_dims(X_test[i], axis=0)
        img = img.astype(np.uint8) if input_dtype == np.uint8 else img.astype(np.float32)

        interpreter.set_tensor(input_details[0]['index'], img)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_details[0]['index'])

        if output_details[0]['dtype'] == np.uint8:
            scale, zero_point = output_details[0]['quantization']
            output_data = (output_data.astype(np.float32) - zero_point) * scale

        predictions_list.append(output_data[0])

    predictions = np.array(predictions_list)
    inf_time = time.time() - start

    y_pred = np.argmax(predictions, axis=1)

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"  Accuracy  : {accuracy:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Inferencia: {inf_time:.2f}s ({len(y_true)} imágenes)")

    # Matriz de confusión → Excel
    cm = confusion_matrix(y_true, y_pred)
    pd.DataFrame(cm, index=eval_class_names, columns=eval_class_names).to_excel(
        excel_writer, sheet_name=f"{model_name}_CM"
    )

    # Reporte → Excel
    report = classification_report(
        y_true, y_pred, target_names=eval_class_names,
        output_dict=True, zero_division=0
    )
    pd.DataFrame(report).transpose().to_excel(
        excel_writer, sheet_name=f"{model_name}_Report"
    )

    eval_results.append({
        "Modelo":          f"{model_name}",
        "Accuracy":        accuracy,
        "Precision":       precision,
        "Recall":          recall,
        "F1 Score":        f1,
        "Loss":            "N/A",
        "Inferencia (s)":  inf_time,
        "Imágenes test":   len(y_true),
        "Clases":          len(eval_class_names)
    })

    del interpreter
    gc.collect()

# Hoja resumen
results_df = pd.DataFrame(eval_results).sort_values("Accuracy", ascending=False)
results_df.to_excel(excel_writer, sheet_name="Summary", index=False)
excel_writer.close()

print("\n" + "="*55)
print("  RESULTADOS FINALES")
print("="*55)
print(results_df[["Modelo", "Accuracy", "F1 Score", "Inferencia (s)"]].to_string(index=False))
print("\n✅ Guardado: model_comparison.xlsx")

Found 21 files belonging to 2 classes.
Clases de evaluación: ['español', 'maya']
Test set cargado: (21, 224, 224, 3) | Etiquetas: (21,)

  Evaluando TFLite: MobileNetV3Large
  Accuracy  : 0.0476
  Precision : 1.0000
  Recall    : 0.0476
  F1 Score  : 0.0909
  Inferencia: 22.46s (21 imágenes)

  RESULTADOS FINALES
          Modelo  Accuracy  F1 Score  Inferencia (s)
MobileNetV3Large  0.047619  0.090909       22.459334

✅ Guardado: model_comparison.xlsx


## 11. Predicción individual

In [20]:
import numpy as np
import tensorflow as tf

def predict_spectrogram_tflite(image_path, tflite_model_path, class_names, img_size=224):
    """
    Predice el idioma de un espectrograma individual usando un modelo .tflite en Python.
    
    image_path        : ruta al PNG del espectrograma
    tflite_model_path : ruta al archivo .tflite (ej. MobileNetV3Large_normal.tflite)
    class_names       : lista de nombres de clase (ej. ['español', 'maya'])
    """
    # 1. Cargar el intérprete de TFLite y reservar la memoria de los tensores
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    # 2. Obtener los detalles de entrada y salida del modelo
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # 3. Cargar la imagen y procesarla exactamente igual que en el entrenamiento
    img = tf.keras.utils.load_img(image_path, target_size=(img_size, img_size))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Agrega la dimensión del batch -> (1, 224, 224, 3)

    # 4. ALINEACIÓN CRÍTICA: Forzar el tipo de dato que espera el modelo (usualmente float32)
    # Esto previene problemas de desalineación de memoria que inflan la confianza
    input_dtype = input_details[0]['dtype']
    img_array = img_array.astype(input_dtype)

    # 5. Inyectar la matriz de pixeles en el tensor de entrada
    interpreter.set_tensor(input_details[0]['index'], img_array)

    # 6. Ejecutar la inferencia en el modelo
    interpreter.invoke()

    # 7. Extraer los resultados (el output del Softmax integrado)
    predictions = interpreter.get_tensor(output_details[0]['index'])[0]

    # 8. Obtener la clase ganadora y su porcentaje
    predicted_idx = np.argmax(predictions)
    confidence = predictions[predicted_idx] * 100

    print(f"Imagen    : {image_path}")
    print(f"Predicción: {class_names[predicted_idx]} (clase {predicted_idx})")
    print(f"Confianza : {confidence:.2f}%")
    print(f"Scores    : {dict(zip(class_names, predictions.tolist()))}")

    return class_names[predicted_idx], confidence


# --- EJECUCIÓN ---
# Nota: Asegúrate de definir tu lista de clases en el orden alfabético correcto
mis_clases = ['español', 'maya']

predict_spectrogram_tflite(
    image_path        = "dataset-entrenamiento/test/español/2_clip_5.png",
    tflite_model_path = "MobileNetV3Large_normal.tflite",
    class_names       = mis_clases
)

Imagen    : dataset-entrenamiento/test/español/2_clip_5.png
Predicción: maya (clase 1)
Confianza : 100.00%
Scores    : {'español': 3.6245296541892458e-06, 'maya': 0.9999964237213135}


('maya', 99.99964237213135)

## 12. Comparación de tamaños de modelo

Vista rápida de cuánto pesa cada archivo generado.

In [6]:
print(f"{'Archivo':<45} {'Tamaño':>10}")
print("-" * 57)

extensions = [".keras", "_fp16.tflite", "_int8.tflite", "_best.keras", "_best_ft.keras"]

for name in MODELS.keys():
    for ext in extensions:
        path = f"{name}{ext}"
        if os.path.exists(path):
            size = os.path.getsize(path) / 1024 / 1024
            print(f"  {path:<43} {size:>8.2f} MB")

print("\n" + "="*57)
print("="*57)

Archivo                                           Tamaño
---------------------------------------------------------
  MobileNetV3Large.keras                         36.16 MB
  MobileNetV3Large_fp16.tflite                    6.36 MB
  MobileNetV3Large_best.keras                    14.64 MB
  MobileNetV3Large_best_ft.keras                 36.16 MB
  MobileNetV2.keras                              28.62 MB
  MobileNetV2_fp16.tflite                         5.08 MB
  MobileNetV2_best.keras                         12.69 MB
  MobileNetV2_best_ft.keras                      28.62 MB

  Guía de selección para móvil:
  • Tiempo real / low-end  → MobileNetV2 INT8
  • Mejor balance          → MobileNetV3Large FP16
  • Máxima precisión       → EfficientNetB0 FP16
